# 05 — Evaluation (your models)

Computes the **spec metrics** on the test set:
top-1 / top-5 accuracy, overall accuracy, macro precision / recall / F1,
confusion-matrix analysis, inference timing, and training curves if available.

**This notebook is for your runs** (from-scratch first).  
Nate trains pretrained on his machine — if he later gives you a compatible `best.pt`, you can point `OPTIONAL_SECOND_CKPT` at it for a side-by-side table. We are **not** building his training notebook here.

**Kernel:** `Group_project\dl_pipeline\.venv`

**Before running the full test eval:** finish the full training cell in `03_train_from_scratch.ipynb` so `checkpoints/resnet18_scratch/best.pt` exists.

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

PIPELINE_ROOT = Path.cwd().resolve()
if PIPELINE_ROOT.name == "notebooks":
    PIPELINE_ROOT = PIPELINE_ROOT.parent
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from src.config import (
    BATCH_SIZE,
    CHECKPOINTS_DIR,
    NUM_CLASSES,
    RESULTS_DIR,
    SEED,
    ensure_output_dirs,
    set_seed,
)
from src.dataset import build_dataloaders, build_datasets, idx_to_category_id
from src.metrics import (
    collect_predictions,
    compute_metrics,
    make_confusion_matrix,
    most_confused_pairs,
)
from src.models import build_model
from src.train_utils import load_checkpoint

set_seed(SEED)
ensure_output_dirs()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

## 2. Checkpoint paths

Primary = **your** from-scratch run.  
Optional second = only if someone hands you a finished `.pt` (leave as `None` for now).

In [ ]:
SCRATCH_CKPT = CHECKPOINTS_DIR / "resnet18_scratch" / "best.pt"
SCRATCH_HISTORY = RESULTS_DIR / "resnet18_scratch_history.json"

# Leave None unless you receive a second trained checkpoint to compare
OPTIONAL_SECOND_CKPT = None  # e.g. Path(r"C:\path\to\someone_elses_best.pt")
OPTIONAL_SECOND_NAME = "other_model"
OPTIONAL_SECOND_PRETRAINED = True  # set to match how that model was trained

EVAL_BATCH = BATCH_SIZE  # 16 is fine for inference on 4GB
USE_AMP = True

print("scratch ckpt exists:", SCRATCH_CKPT.is_file(), "→", SCRATCH_CKPT)
if not SCRATCH_CKPT.is_file():
    print(
        "\n⚠ No scratch checkpoint yet. Run the FULL training cell (§5) in "
        "03_train_from_scratch.ipynb first, then re-run this notebook."
    )

## 3. Data (test set only for final numbers)

In [ ]:
train_ds, val_ds, test_ds = build_datasets(augment_train=False)
_, _, test_loader = build_dataloaders(
    train_ds, val_ds, test_ds, batch_size=EVAL_BATCH
)
cat_id = idx_to_category_id(train_ds)
labels = list(range(NUM_CLASSES))
print(f"test images={len(test_ds)}  classes={NUM_CLASSES}")

## 4. Load model(s) and run test predictions

In [ ]:
def load_resnet18_for_eval(ckpt_path: Path, *, pretrained_flag: bool) -> tuple:
    """Rebuild ResNet-18 and load weights. pretrained_flag only affects init before load."""
    model = build_model("resnet18", num_classes=NUM_CLASSES, pretrained=False)
    ckpt = load_checkpoint(ckpt_path, model, map_location=device)
    model.to(device)
    model.eval()
    meta = {
        "epoch": ckpt.get("epoch"),
        "best_val_acc": ckpt.get("best_val_acc"),
        "config": ckpt.get("config", {}),
        "pretrained_flag": pretrained_flag,
    }
    return model, meta


def eval_one(name: str, ckpt_path: Path, *, pretrained_flag: bool) -> dict:
    print(f"\n=== Evaluating: {name} ===")
    model, meta = load_resnet18_for_eval(ckpt_path, pretrained_flag=pretrained_flag)
    print(f"ckpt epoch={meta['epoch']}  stored best_val_acc={meta['best_val_acc']}")

    preds = collect_predictions(model, test_loader, device, use_amp=USE_AMP, topk=5)
    metrics = compute_metrics(
        preds["y_true"], preds["y_pred"], preds["y_topk"], labels=labels
    )
    metrics["inference_logits_seconds"] = preds["logits_time"]
    metrics["inference_wall_seconds"] = preds["wall_time"]
    metrics["inference_img_per_sec"] = preds["n_samples"] / max(preds["logits_time"], 1e-8)
    metrics["n_test"] = preds["n_samples"]

    cm = make_confusion_matrix(preds["y_true"], preds["y_pred"], labels=labels)
    confused = most_confused_pairs(cm, cat_id, top_n=15)

    # free GPU before next model
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "name": name,
        "meta": meta,
        "metrics": metrics,
        "cm": cm,
        "confused": confused,
        "y_true": preds["y_true"],
        "y_pred": preds["y_pred"],
    }


results = []
assert SCRATCH_CKPT.is_file(), f"Missing {SCRATCH_CKPT} — finish notebook 03 full training first."
results.append(eval_one("resnet18_scratch", SCRATCH_CKPT, pretrained_flag=False))

if OPTIONAL_SECOND_CKPT is not None and Path(OPTIONAL_SECOND_CKPT).is_file():
    results.append(
        eval_one(
            OPTIONAL_SECOND_NAME,
            Path(OPTIONAL_SECOND_CKPT),
            pretrained_flag=OPTIONAL_SECOND_PRETRAINED,
        )
    )
else:
    print("\n(No optional second checkpoint — scratch-only eval.)")

## 5. Metrics table (report-ready)

In [ ]:
rows = []
for r in results:
    m = r["metrics"]
    rows.append(
        {
            "model": r["name"],
            "top1_%": m["top1_acc"] * 100,
            "top5_%": m["top5_acc"] * 100,
            "macro_P": m["macro_precision"],
            "macro_R": m["macro_recall"],
            "macro_F1": m["macro_f1"],
            "infer_img/s": m["inference_img_per_sec"],
            "infer_s": m["inference_logits_seconds"],
        }
    )

metrics_df = pd.DataFrame(rows).round(4)
display(metrics_df)

out_csv = RESULTS_DIR / "test_metrics.csv"
metrics_df.to_csv(out_csv, index=False)
print(f"saved {out_csv}")

## 6. Training curves (scratch)

Loaded from the JSON written by notebook 03.

In [ ]:
if SCRATCH_HISTORY.is_file():
    with SCRATCH_HISTORY.open() as f:
        payload = json.load(f)
    hist = payload["history"]
    epochs = range(1, len(hist["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, hist["train_loss"], label="train")
    axes[0].plot(epochs, hist["val_loss"], label="val")
    axes[0].set_title("Scratch — loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, [a * 100 for a in hist["train_acc"]], label="train")
    axes[1].plot(epochs, [a * 100 for a in hist["val_acc"]], label="val")
    axes[1].set_title("Scratch — top-1 acc (%)")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

    cfg = payload.get("config", {})
    if "total_train_seconds" in cfg:
        print(f"total train time: {cfg['total_train_seconds']/60:.1f} min")
else:
    print(f"No history JSON at {SCRATCH_HISTORY} (finish notebook 03 full run).")

## 7. Confusion matrix (subset view)

Full 500×500 is unreadable as a heatmap. We plot:
1. A **20-class slice** (first 20 indices) for a visual check
2. The **top confused category-id pairs** (table) for the report error analysis

In [ ]:
scratch = results[0]
cm = scratch["cm"]

SLICE = 20
cm_slice = cm[:SLICE, :SLICE]
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm_slice, ax=ax, cmap="Blues", cbar=True)
ax.set_title(f"Confusion matrix slice [{SLICE}×{SLICE}] — {scratch['name']}")
ax.set_xlabel("predicted idx")
ax.set_ylabel("true idx")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "scratch_cm_slice.png", dpi=150)
plt.show()

print("Most confused pairs (true cat_id → pred cat_id, count):")
for true_n, pred_n, cnt in scratch["confused"]:
    print(f"  {true_n:>6} → {pred_n:<6}  {cnt}")

np.save(RESULTS_DIR / "scratch_cm_full.npy", cm)
print(f"\nFull CM saved to {RESULTS_DIR / 'scratch_cm_full.npy'}")

## 8. Done (your side)

You now have report artefacts under `results/`:
- `test_metrics.csv`
- `scratch_cm_slice.png` + `scratch_cm_full.npy`
- curves from history JSON

**Still on you:** let the full `03` training finish if it has not.  
**Nate:** his pretrained training + Grad-CAM on his machine — not in this notebook.